# **TUTORIAL**: Study objective B0, dynamic CC (uncertainty)

AESA Phase B: (dynamic) carrying capacities definition.\
Run aSoCC Monte Carlo uncertainty on extracted AR6 climate change pathways from `process_ar6(...)` outputs.

# Before starting...

### Set workspace

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")

# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

### Run prerequisites

This tutorial assumes that the workspace has already been set, with all prerequisites available (downloads and processing).\
If this is not your case, it is recommended to first go through the core prerequisite notebooks: [tutorials/core_prerequisites/0_set_workspace.ipynb](../../core_prerequisites/0_set_workspace.ipynb), [tutorials/core_prerequisites/1_download_data.ipynb](../../core_prerequisites/1_download_data.ipynb), and [tutorials/core_prerequisites/2_process_data.ipynb](../../core_prerequisites/2_process_data.ipynb).

# Basic features of the uncertainty function

### In a nutshell

The function takes necessary **inputs**, mainly for two purposes:
- inputs for the deterministic function `base_ar6_cc_args`, including `years`, `category`, `ssp_scenario`, and dynamic carrying capacity emission route parameters.
- inputs for the configuration of uncertainty analysis (Monte Carlo) through `uncertainty_config`. The AR6 CC uncertainty source list contains `dynamic_ar6_cc_uncertainty`, plus `mc_parameters` for the Monte Carlo run policy. `dynamic_ar6_cc_uncertainty` is active by default; its nested `category_uncertainty` option is inactive by default and must be requested explicitly.

Set `category_uncertainty=True` when Monte Carlo should sample one AR6 climate category uniformly across the requested categories with retained model-scenario pairs, then sample a retained model-scenario pair within that category pool.

The uncertainty **output** folder contains:
- Monte Carlo run values and summary statistics.
- `results/README.txt` explains the public row identity tables, run value fragments, run interval indexes, summary tables, source method logs, and compact CSV or Parquet reading contract.
- **Figures** are rendered by default (`figures=True`). Use `figures=False` to skip them; `figure_format` controls the file format and resolution.
- log files, including uncertainty source parameter logs.

Basic features also involve:
- **overwriting** of existing values: use the `refresh` parameter.

Methodological details on AR6 scenario filtering, harmonization, gross emissions modes, and dynamic carrying capacity construction are documented in <a href="../../methodological_notes/methodological_note__steady_state__dynamic_cc.pdf">methodological_notes/methodological_note__steady_state__dynamic_cc.pdf</a>.


### Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>uncertainty_ar6_cc(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>base_ar6_cc_args</code></td><td>Deterministic AR6 CC selector envelope written as a<br>&#10;dictionary. Required key: <code>years</code>. Accepted optional keys are<br>&#10;<code>harmonization</code>, <code>harmonization_method</code>, <code>category</code>,<br>&#10;<code>ssp_scenario</code>, <code>emission_type</code>, <code>include_afolu</code>,<br>&#10;<code>emissions_mode</code>, and <code>subset_version</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>years</code>: Study year selector provided as a consecutive year<br>&#10;  list or <code>range(start_year, end_year + 1)</code>. The resolved years<br>&#10;  must contain at least two consecutive years with no gaps.<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>harmonization</code>: Whether to harmonize retained AR6 pathways<br>&#10;  to the historical baseline. <strong>Defaults to</strong> <code>True</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>harmonization_method</code>: Harmonization method applied only when<br>&#10;  <code>harmonization=True</code>. <strong>Defaults to</strong> <code>&quot;offset&quot;</code>. The only<br>&#10;  supported value is currently <code>&quot;offset&quot;</code>.<br>&#10;  Ignored when <code>harmonization=False</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>category</code>: AR6 category classification selector for global<br>&#10;  warming trajectories. Accepts a string such as <code>&quot;C3&quot;</code> or a list such as<br>&#10;  <code>[&quot;C1&quot;, &quot;C2&quot;]</code>. Valid values are <code>&quot;C1&quot;</code> through <code>&quot;C8&quot;</code>.<br>&#10;  <strong>Defaults to</strong> <code>[&quot;C1&quot;, &quot;C2&quot;, &quot;C3&quot;, &quot;C4&quot;]</code>, the categories<br>&#10;  aligned with the 2015 Paris Agreement.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>ssp_scenario</code>: SSP label selector as a string, list, or<br>&#10;  <code>None</code>. <strong>Defaults to</strong><br>&#10;  <code>[&quot;SSP1&quot;, &quot;SSP2&quot;, &quot;SSP3&quot;, &quot;SSP4&quot;, &quot;SSP5&quot;]</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>emission_type</code>: Dynamic AR6 emission type. Accepted values<br>&#10;  are <code>&quot;kyoto_gases&quot;</code> (<strong>default</strong>) and <code>&quot;co2&quot;</code>.<br>&#10;  <code>emission_type=&quot;kyoto_gases&quot;</code> uses the GWP100 Kyoto Gases<br>&#10;  aggregate; <code>emission_type=&quot;co2&quot;</code> uses direct CO2 pathways.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>include_afolu</code>: Whether AFOLU emissions are included inside<br>&#10;  the selected <code>emission_type</code>. <strong>Defaults to</strong> <code>False</code>.</span><br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>emissions_mode</code>: Dynamic AR6 emissions mode. Accepted<br>&#10;  values are <code>&quot;net&quot;</code>, <code>&quot;gross&quot;</code>, and <code>&quot;gross_alt&quot;</code>.<br>&#10;  <strong>Defaults to</strong> <code>&quot;gross_alt&quot;</code>. <code>&quot;net&quot;</code> uses net AR6 emissions<br>&#10;  pathways directly. <code>&quot;gross&quot;</code> removes all sequestration<br>&#10;  sources from net emissions. <code>&quot;gross_alt&quot;</code> removes all<br>&#10;  sequestration sources except CCS. CCS is retained because<br>&#10;  IPCC AR6 treats CCS as capture at fossil or industrial point<br>&#10;  sources rather than direct removal of CO2 from the atmosphere,<br>&#10;  so it is kept separate from net negative sequestration. Gross<br>&#10;  modes write positive emissions rows and signed negative<br>&#10;  sequestration companion rows; downstream aCC and ASR consume<br>&#10;  only the positive emissions rows. See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__steady_state__dynamic_cc.pdf</code><br>&#10;  for the methodological explanation.</span><br>&#10;<span style="color:#c45f00">&bull;&nbsp;<code>subset_version</code>: Optional selector for a subset of AR6<br>&#10;  model-scenario pairs. Follow<br>&#10;  <code>data_processed/ar6/&lt;processed_scope&gt;/README_model_scenario_subset.txt</code><br>&#10;  to create the subset CSV. <strong>Defaults to</strong> <code>None</code>.</span></td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>uncertainty_config</code></td><td>Monte Carlo configuration dictionary. The <strong>default</strong><br>&#10;signature activates dynamic AR6 CC uncertainty. Category<br>&#10;uncertainty is inactive by <strong>default</strong>. Source blocks use an<br>&#10;<code>active</code> boolean; write <code>active=False</code> to disable dynamic AR6<br>&#10;CC uncertainty. See<br>&#10;<code>data_raw/methodological_notes/methodological_note__acc_uncertainty_sources.pdf</code><br>&#10;for uncertainty source definitions and mathematical expressions.<br>&#10;&#160;<br>&#10;Accepted keys:<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>mc_parameters</code>: optional dictionary with <code>convergence</code> and<br>&#10;  <code>fixed</code> mode blocks. Exactly one mode must be active.<br>&#10;&#160;<br>&#10;  Nested mode blocks:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull;&nbsp;<code>convergence</code>: convergence mode block. This is the <strong>default</strong><br>&#10;    active mode.<br>&#10;&#160;<br>&#10;    Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">    &bull;&nbsp;<code>active</code>: Whether convergence mode is active.</span><br>&#10;<span style="color:#087f5b">    &bull;&nbsp;<code>max_runs</code>: Maximum number of Monte Carlo runs allowed<br>&#10;      before stopping.</span><br>&#10;<span style="color:#087f5b">    &bull;&nbsp;<code>rtol</code>: Relative tolerance used to decide whether<br>&#10;      monitored summary statistics have converged.</span><br>&#10;<span style="color:#087f5b">    &bull;&nbsp;<code>stable_runs</code>: Number of consecutive accepted runs that<br>&#10;      must remain within tolerance before the run stops.</span><br>&#10;<span style="color:#087f5b">    &bull;&nbsp;<code>convergence_statistics</code>: Statistic monitored for<br>&#10;      convergence. Monte Carlo convergence is mean only and<br>&#10;      <strong>defaults to</strong> <code>[&quot;mean&quot;]</code>.</span><br>&#10;&#160;<br>&#10;<span style="color:#c45f00">  &bull;&nbsp;<code>fixed</code>: fixed run count mode block.<br>&#10;&#160;<br>&#10;    Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">    &bull;&nbsp;<code>active</code>: Whether fixed mode is active.</span><br>&#10;    &bull;&nbsp;<code>n_runs</code>: Exact number of Monte Carlo runs.<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull;&nbsp;<code>dynamic_ar6_cc_uncertainty</code>: optional AR6 CC source block. It<br>&#10;  <strong>defaults to</strong> <code>{&quot;active&quot;: True, &quot;sampling_method&quot;: &quot;srs&quot;, &quot;category_uncertainty&quot;: False}</code>. <code>sampling_method</code> accepts<br>&#10;  <code>&quot;srs&quot;</code> for simple random sampling (samples across retained<br>&#10;  model-scenario pairs matching the requested category and SSP) or<br>&#10;  <code>&quot;lhs&quot;</code> for Latin hypercube sampling (samples among retained<br>&#10;  models first, then among retained scenarios for the selected<br>&#10;  model, category, and SSP to limit over representation of models<br>&#10;  with more AR6 submissions). The effect of this choice is visible<br>&#10;  in <code>process_ar6(...)</code> sampling diagnostic figures.<br>&#10;  <code>category_uncertainty</code> is inactive by <strong>default</strong>. If <code>True</code>,<br>&#10;  each Monte Carlo run first samples one retained AR6 category<br>&#10;  with equal probability in the studied SSP pool. It then applies<br>&#10;  <code>sampling_method</code> inside that selected category; with<br>&#10;  <code>sampling_method=&quot;lhs&quot;</code>, this means model first, then scenario<br>&#10;  inside that model.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull;&nbsp;<code>active</code>: Whether dynamic AR6 carrying capacity uncertainty<br>&#10;    is active.</span><br>&#10;<span style="color:#087f5b">  &bull;&nbsp;<code>sampling_method</code>: AR6 pathway sampling method. Accepted</span><br>&#10;<span style="color:#087f5b">    values are <code>&quot;srs&quot;</code></span> and <code>&quot;lhs&quot;</code>.<br>&#10;<span style="color:#c45f00">  &bull;&nbsp;<code>category_uncertainty</code>: Whether each Monte Carlo run samples<br>&#10;    one retained AR6 category before applying <code>sampling_method</code><br>&#10;    inside that category.</span></td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>output_format</code></td><td>Public uncertainty table format, either<br>&#10;<code>&quot;csv_compact&quot;</code> or <code>&quot;parquet&quot;</code>. <strong>Defaults to</strong><br>&#10;<code>&quot;csv_compact&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figures</code></td><td>Whether to render figures.<br>&#10;<strong>Default is</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull;&nbsp;<code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>refresh</code></td><td>If <code>True</code>, refresh the matching processed AR6 output scope selected by <code>process_ar6(...)</code>, the resolved deterministic AR6 CC output scope, and the resolved AR6 CC Monte Carlo outputs for this uncertainty request. The Monte Carlo refresh removes matching run folders for the current request under the adjacent <code>monte_carlo</code> root of that deterministic output scope. For example, matching AR6 CC Monte Carlo run folders are refreshed under <code>&lt;repo&gt;/data_processed/ar6/2019-2060_harmonization_offset/ar6_cc/gross_alt_kyoto_gases_wo_afolu/C1__SSP1/monte_carlo/mc_&lt;generated_id&gt;</code>. Raw downloads and downstream aCC or ASR outputs are not refreshed. <strong>Defaults to</strong> <code>False</code>.</td></tr>
</tbody>
</table>

</details>


### Dynamic AR6 CC Monte Carlo in default convergence mode

In [ ]:
from pyaesa import uncertainty_ar6_cc

uncertainty_ar6_cc(
    base_ar6_cc_args={
        "years": range(2020, 2051),
        "category": ["C1", "C2"],
        "ssp_scenario": ["SSP2"],
    },
    uncertainty_config={
        # mc_parameters is omitted, so Monte Carlo uses convergence mode defaults.
        # To change convergence tolerance or maximum run count, add for example:
        # "mc_parameters": {"convergence": {"rtol": 0.02, "max_runs": 100000}},
        "dynamic_ar6_cc_uncertainty": {
            "category_uncertainty": True,
        },
    },
)

# What to do next

**Beginners**

If you are a new user in the process of discovering <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span>, it is recommend that you first discover all study objectives with the **basic features** available.
Have a look at the other notebooks available at [tutorials/study_objectives/0_study_objectives.md](../0_study_objectives.md)

**Experts**

If you are already familiar with <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> and if you want to discover **advanced features** available, check out examples in the next section of this tutorial !

# Advanced features

Advanced features currently available include:
- Changing the sampling method for the Monte Carlo simulation
- Changing convergence tolerance or maximum runs

### Changing the sampling method for the Monte Carlo simulation

`dynamic_ar6_cc_uncertainty["sampling_method"]` selects how retained AR6
model-scenario rows are sampled.

- `"srs"` is the default simple random sampling across retained model-scenario
  pairs matching the requested category and SSP filters.
- `"lhs"` samples a model first, then samples one retained scenario for that
  model, category, and SSP. This limits over representation of models with more
  AR6 submissions.

`category_uncertainty=True` can be combined with either sampling method. In
that case each Monte Carlo run first samples one retained AR6 category, then
samples a model-scenario row inside that category.

In [ ]:
uncertainty_ar6_cc(
    base_ar6_cc_args={
        "years": range(2020, 2051),
        "category": ["C1", "C2"],
        "ssp_scenario": ["SSP2"],
    },
    uncertainty_config={
        "mc_parameters": {
            "fixed": {"active": True, "n_runs": 1000},
            "convergence": {"active": False},
        },
        "dynamic_ar6_cc_uncertainty": {
            "sampling_method": "lhs",
            "category_uncertainty": True,
        },
    },
)

### Changing convergence tolerance or maximum runs

`mc_parameters["convergence"]` controls convergence mode. Omit `mc_parameters`
when the default convergence settings are sufficient; write the block only when
the study needs a different `rtol` or `max_runs` value.

In [ ]:
uncertainty_ar6_cc(
    base_ar6_cc_args={
        "years": range(2020, 2051),
        "category": ["C1", "C2"],
        "ssp_scenario": ["SSP2"],
    },
    uncertainty_config={
        "mc_parameters": {
            "convergence": {
                "rtol": 0.02,
            },
        },
    },
)